## Preparing the Dataset

In [39]:
# checking venv

import sys
print(sys.executable)

/home/gusbeihel@GU.GU.SE/ml2-final-project/.venv/bin/python


In [40]:
import json
import pandas as pd

In [41]:
# loading the original data (not clean JSONL)

def load_json_objects(filepath):
    with open(filepath, encoding="utf-8") as f:
        content = f.read()

    decoder = json.JSONDecoder()
    records = []
    position = 0

    while position < len(content):
        # Skip whitespace between objects
        while position < len(content) and content[position].isspace():
            position += 1

        if position >= len(content):
            break

        obj, position = decoder.raw_decode(content, position)
        records.append(obj)

    return pd.DataFrame(records)

train_split = load_json_objects("../data/train.jsonl")
dev_split = load_json_objects("../data/dev.jsonl")
test_split = load_json_objects("../data/test.jsonl")

print("Train:", train_split.shape)
print("Dev:", dev_split.shape)
print("Test:", test_split.shape)

display(train_split.head())

Train: (384, 11)
Dev: (103, 11)
Test: (484, 11)


,author,question_ending,question,preceding_turns,paragraph,paragraph_with_author_and_marker,full_turn,full_turn_with_marker,following_turns,subreddit,label
0,Person_B,why not,"Are we entitled, for example, to require that ...",Person_A: Alexander Hamilton once stressed the...,"Furthermore, if we are allowed to have additio...","Person_B: Furthermore, if we are allowed to ha...","Person_B: Sure, but what is that basis? Why th...","Person_B: Sure, but what is that basis? Why th...","Person_C: > &gt; Are we entitled, for example,...",changemyview,informational
1,Person_B,why,Why?,Person_A: Pretty much what it says in the titl...,&gt;and taxes are the natural cost of having a...,Person_B: &gt;and taxes are the natural cost o...,Person_B: &gt;I enjoy paying my taxes\n\nCan't...,Person_B: &gt;I enjoy paying my taxes\n\nCan't...,"Person_C: > &gt; Yes, but you're assuming gove...",changemyview,informational
2,Person_C,how,How?,Person_A: And how many people answered?\nPerso...,How? It is a matter of probability. Suppose a ...,Person_C: <ques>How?</ques> It is a matter of ...,"Person_C: If a country has a million people, a...","Person_C: If a country has a million people, a...",Person_A: > Of course if its based on random p...,europe,rhetorical
3,Person_B,how,And that changes things how?,"Person_A: The 2% obligation is not in place, y...",Person_B: And that changes things how? The US ...,Person_B: <ques>And that changes things how?</...,Person_B: And that changes things how? The US ...,Person_B: <ques>And that changes things how?</...,Person_C: > Let's take the world's most overbl...,europe,rhetorical
4,Person_B,why,Why?,Person_A: There is no conflict with the exampl...,Person_B: &gt;It's certainly possible. It's j...,Person_B: &gt;It's certainly possible. It's j...,Person_B: &gt;It's certainly possible. It's j...,Person_B: &gt;It's certainly possible. It's j...,Person_A: > Because where do you draw the line...,changemyview,informational


In [42]:
# converting the data to proper JSONL

train_split.to_json("../data/train_clean.jsonl", orient="records", lines=True, force_ascii=False)
dev_split.to_json("../data/dev_clean.jsonl", orient="records", lines=True, force_ascii=False)
test_split.to_json("../data/test_clean.jsonl", orient="records", lines=True, force_ascii=False)

train_split = pd.read_json("../data/train_clean.jsonl", lines=True)
dev_split = pd.read_json("../data/dev_clean.jsonl", lines=True)
test_split = pd.read_json("../data/test_clean.jsonl", lines=True)

I will use the preceding turn as my context condition for each question.

In [63]:
# empty preceding turns
print(train_split["preceding_turns"].isna().sum())

# preceding turns with empty string
print(train_split["preceding_turns"].str.strip().eq("").sum())

# non-textual content in preceding turns
print(train_split["preceding_turns"].str.contains(
    "<non-textual content>", regex=False
).sum())

# ... only non-textual content
print(train_split["preceding_turns"].str.fullmatch(
    r"\s*Person_\w+:\s*<non-textual content>\s*"
).sum())

# checking for [removed] or [deleted]
print(train_split["preceding_turns"].str.fullmatch(
    r"\s*Person_\w+:\s*(?:\[removed\]|\[deleted\])\s*"
).sum())

0
21
48
18
0


In [62]:
# empty preceding turns
print(dev_split["preceding_turns"].isna().sum())

# preceding turns with empty string
print(dev_split["preceding_turns"].str.strip().eq("").sum())

# non-textual content in preceding turns
print(dev_split["preceding_turns"].str.contains(
    "<non-textual content>", regex=False
).sum())

# ... only non-textual content
print(dev_split["preceding_turns"].str.fullmatch(
    r"\s*Person_\w+:\s*<non-textual content>\s*"
).sum())

# checking for [removed] or [deleted]
print(dev_split["preceding_turns"].str.fullmatch(
    r"\s*Person_\w+:\s*(?:\[removed\]|\[deleted\])\s*"
).sum())

0
10
8
3
0


In [64]:
# empty preceding turns
print(test_split["preceding_turns"].isna().sum())

# preceding turns with empty string
print(test_split["preceding_turns"].str.strip().eq("").sum())

# non-textual content in preceding turns
print(test_split["preceding_turns"].str.contains(
    "<non-textual content>", regex=False
).sum())

# ... only non-textual content
print(test_split["preceding_turns"].str.fullmatch(
    r"\s*Person_\w+:\s*<non-textual content>\s*"
).sum())

# checking for [removed] or [deleted]
print(test_split["preceding_turns"].str.fullmatch(
    r"\s*Person_\w+:\s*(?:\[removed\]|\[deleted\])\s*"
).sum())

0
27
51
17
0


Based on the results above, I will filter out preceding turns that:
- are empty strings (with or without whitespaces)
- are only non-textual content
  
This should exclude 96 examples in total.

In [70]:
# original label distribution

print("train:\n", train_split["label"].value_counts(normalize=True))
print("dev:\n", dev_split["label"].value_counts(normalize=True))
print("test:\n", test_split["label"].value_counts(normalize=True))

train:
 label
rhetorical       0.59375
informational    0.40625
Name: proportion, dtype: float64
dev:
 label
rhetorical       0.601942
informational    0.398058
Name: proportion, dtype: float64
test:
 label
rhetorical       0.659091
informational    0.340909
Name: proportion, dtype: float64


In [ ]:
# filtering train

empty = train_split["preceding_turns"].str.strip().eq("")

non_textual = train_split["preceding_turns"].str.fullmatch(
    r"\s*Person_\w+:\s*<non-textual content>\s*"
)

train_filtered = train_split[~(empty | non_textual)].copy()

train_filtered["label"].value_counts(normalize=True)

label
rhetorical       0.611594
informational    0.388406
Name: proportion, dtype: float64

In [ ]:
# filtering dev

empty = dev_split["preceding_turns"].str.strip().eq("")

non_textual = dev_split["preceding_turns"].str.fullmatch(
    r"\s*Person_\w+:\s*<non-textual content>\s*"
)

dev_filtered = dev_split[~(empty | non_textual)].copy()

dev_filtered["label"].value_counts(normalize=True)

label
rhetorical       0.644444
informational    0.355556
Name: proportion, dtype: float64

In [ ]:
# filtering test

empty = test_split["preceding_turns"].str.strip().eq("")

non_textual = test_split["preceding_turns"].str.fullmatch(
    r"\s*Person_\w+:\s*<non-textual content>\s*"
)

test_filtered = test_split[~(empty | non_textual)].copy()

test_filtered["label"].value_counts(normalize=True)

label
rhetorical       0.681818
informational    0.318182
Name: proportion, dtype: float64

Filtering removed more informational questions than rhetorical questions.

In [ ]:
# inspecting context lengths